# Second-Order Optimizers + SDP on Continual RL (PPO - Ant-v4)
#
**Hypothesis**: SDP + second-order optimizers mitigate Loss of Plasticity
in RL, just as they do in supervised continual learning (Permuted MNIST).
#
**Optimizers tested**:
1. **AdaHessian** (Yao et al., 2021) - Hutchinson diagonal Hessian
2. **SophiaH** (Liu et al., 2023) - Hutchinson Hessian + element-wise clipping
3. **Shampoo** (Gupta et al., 2018) - Full-matrix preconditioning
4. **ASAM** (Kwon et al., 2021) - Adaptive SAM + SGD base
5. **SASSHA** (baseline) - SAM + Hutchinson Hessian
#
Each optimizer runs **with and without SDP** for ablation.
#
**Benchmark**: PPO on Ant-v4 (MuJoCo), 10M steps.
Network: MLP 27→256×2→8 (policy) + 27→256×2→1 (value), ReLU.
Based on CBP paper RL setup (lop/rl/) and rl-continual.ipynb.

## 1. Imports & Setup

In [1]:
pip install gymnasium[mujoco] mujoco scipy matplotlib tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 64.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 243.5/243.5 kB 21.3 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os, sys, time, pickle, math, copy
from copy import deepcopy
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Optimizer
from tqdm import tqdm
import matplotlib
import matplotlib.pyplot as plt
matplotlib.rcParams['font.family'] = 'serif'
matplotlib.rcParams['font.size'] = 11

import gymnasium as gym

_LOP_ROOT = "/kaggle/input/datasets/caothianhtho/lop-src"
if os.path.isdir(_LOP_ROOT) and _LOP_ROOT not in sys.path:
    sys.path.insert(0, _LOP_ROOT)

from lop.nets.policies import MLPPolicy
from lop.nets.valuefs import MLPVF
from lop.algos.rl.buffer import Buffer
from lop.utils.miscellaneous import compute_matrix_rank_summaries

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


## 2. Metrics & SDP for RL

In [3]:
@torch.no_grad()
def compute_dormant_units_rl(pol, vf, threshold=0.01):
    """Compute fraction of dormant neurons from stored activations."""
    total_units, dormant_units = 0, 0
    for net in [pol, vf]:
        if hasattr(net, 'activations') and net.activations:
            for key, feat in net.activations.items():
                if feat is not None and feat.dim() >= 2:
                    activity = (feat != 0).float().mean(dim=0)
                    dormant = (activity < threshold).sum().item()
                    dormant_units += dormant
                    total_units += activity.numel()
    return dormant_units / total_units if total_units > 0 else 0.0


@torch.no_grad()
def compute_stable_rank_from_features(feature_activity):
    """Compute stable rank from feature activations (99% variance)."""
    if feature_activity is None or feature_activity.numel() == 0:
        return 1.0
    _, _, _, stable_rank = compute_matrix_rank_summaries(
        m=feature_activity, prop=0.99, use_scipy=True
    )
    return stable_rank.item() if torch.is_tensor(stable_rank) else float(stable_rank)


@torch.no_grad()
def compute_weight_magnitude(pol, vf):
    """Compute average weight magnitude across both networks."""
    total, n = 0.0, 0
    for net in [pol, vf]:
        for name, p in net.named_parameters():
            if 'weight' in name:
                total += p.abs().mean().item()
                n += 1
    return total / n if n else 0.0


def apply_sdp_rl(pol, vf, gamma):
    """
    Spectral Diversity Preservation (SDP) for RL actor-critic.
    σ'_i = σ̄^γ · σ_i^(1-γ)

    Applied to Linear layers of both policy and value networks.
    Skips output layers (structural rank bottlenecks).
    """
    cond_numbers = []
    for net_name, net in [('pol', pol), ('vf', vf)]:
        # Determine which Sequential to process
        if hasattr(net, 'mean_net'):
            seq = net.mean_net
        elif hasattr(net, 'v_net'):
            seq = net.v_net
        else:
            continue
        modules = [m for m in seq.modules() if isinstance(m, nn.Linear)]
        with torch.no_grad():
            for i, module in enumerate(modules):
                is_output = (i == len(modules) - 1)
                if is_output:
                    continue  # skip output layer
                W = module.weight.data
                try:
                    U, S, Vh = torch.linalg.svd(W, full_matrices=False)
                except Exception:
                    continue
                if S.numel() == 0 or S[0] < 1e-12:
                    continue
                cond_numbers.append((S[0] / S[-1].clamp(min=1e-12)).item())
                s_mean = S.mean()
                S_new = (s_mean ** gamma) * (S ** (1.0 - gamma))
                W_new = U @ torch.diag(S_new) @ Vh
                module.weight.data.copy_(W_new)
    return cond_numbers


print("✓ Metrics & SDP defined")

✓ Metrics & SDP defined


## 3. Optimizer Definitions

### 3a. AdaHessian

In [ ]:
class Adahessian(Optimizer):
    """AdaHessian - Adaptive second-order optimizer using Hutchinson trace."""
    def __init__(self, params, lr=0.15, betas=(0.9, 0.999), eps=1e-4,
                 weight_decay=0.0, hessian_power=1, lazy_hessian=1,
                 n_samples=1, seed=0):
        if not 0.0 <= lr: raise ValueError(f"Invalid lr: {lr}")
        if not 0.0 <= eps: raise ValueError(f"Invalid eps: {eps}")
        if not 0.0 <= betas[0] < 1.0: raise ValueError(f"Invalid beta0: {betas[0]}")
        if not 0.0 <= betas[1] < 1.0: raise ValueError(f"Invalid beta1: {betas[1]}")
        if not 0.0 <= hessian_power <= 1.0: raise ValueError(f"Invalid hessian_power: {hessian_power}")
        self.n_samples = n_samples
        self.lazy_hessian = lazy_hessian
        self.seed = seed
        self.generator = torch.Generator().manual_seed(self.seed)
        defaults = dict(lr=lr, betas=betas, eps=eps, weight_decay=weight_decay,
                        hessian_power=hessian_power)
        super().__init__(params, defaults)
        for p in self.get_params():
            p.hess = 0.0
            self.state[p]["hessian step"] = 0

    def get_params(self):
        return (p for group in self.param_groups for p in group['params'] if p.requires_grad)

    def zero_hessian(self):
        for p in self.get_params():
            if not isinstance(p.hess, float) and self.state[p]["hessian step"] % self.lazy_hessian == 0:
                p.hess.zero_()

    @torch.no_grad()
    def set_hessian(self):
        params = []
        for p in filter(lambda p: p.grad is not None, self.get_params()):
            if self.state[p]["hessian step"] % self.lazy_hessian == 0:
                params.append(p)
            self.state[p]["hessian step"] += 1
        if len(params) == 0: return
        if self.generator.device != params[0].device:
            self.generator = torch.Generator(params[0].device).manual_seed(self.seed)
        grads = [p.grad for p in params]
        for i in range(self.n_samples):
            zs = [torch.randint(0, 2, p.size(), generator=self.generator, device=p.device) * 2.0 - 1.0 for p in params]
            h_zs = torch.autograd.grad(grads, params, grad_outputs=zs, only_inputs=True,
                                       retain_graph=i < self.n_samples - 1)
            for h_z, z, p in zip(h_zs, zs, params):
                p.hess += h_z * z / self.n_samples

    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None: loss = closure()
        self.zero_hessian()
        self.set_hessian()
        for group in self.param_groups:
            for p in group['params']:
                if p.grad is None or p.hess is None: continue
                if p.dim() <= 2:
                    p.hess = p.hess.abs().clone()
                if p.dim() == 4:
                    p.hess = torch.mean(p.hess.abs(), dim=[2, 3], keepdim=True).expand_as(p.hess).clone()
                p.mul_(1 - group['lr'] * group['weight_decay'])
                state = self.state[p]
                if len(state) == 1:
                    state['step'] = 0
                    state['exp_avg'] = torch.zeros_like(p.data)
                    state['exp_hessian_diag_sq'] = torch.zeros_like(p.data)
                exp_avg, exp_hessian_diag_sq = state['exp_avg'], state['exp_hessian_diag_sq']
                beta1, beta2 = group['betas']
                state['step'] += 1
                exp_avg.mul_(beta1).add_(p.grad, alpha=1 - beta1)
                exp_hessian_diag_sq.mul_(beta2).addcmul_(p.hess, p.hess, value=1 - beta2)
                bias_correction1 = 1 - beta1 ** state['step']
                bias_correction2 = 1 - beta2 ** state['step']
                step_size = group['lr'] / bias_correction1
                k = group['hessian_power']
                denom = (exp_hessian_diag_sq / bias_correction2).pow_(k / 2).add_(group['eps'])
                p.addcdiv_(exp_avg, denom, value=-step_size)
        return loss

print("✓ AdaHessian defined")

✓ AdaHessian defined


### 3b. SophiaH

In [ ]:
class SophiaH(Optimizer):
    """SophiaH - Hutchinson Hessian + element-wise clipping."""
    def __init__(self, params, lr=0.15, betas=(0.965, 0.99), eps=1e-15,
                 weight_decay=1e-1, lazy_hessian=10, n_samples=1,
                 clip_threshold=0.04, seed=0):
        if not 0.0 <= lr: raise ValueError(f"Invalid lr: {lr}")
        self.n_samples = n_samples
        self.lazy_hessian = lazy_hessian
        self.seed = seed
        self.generator = torch.Generator().manual_seed(self.seed)
        defaults = dict(lr=lr, betas=betas, eps=eps, weight_decay=weight_decay,
                        clip_threshold=clip_threshold)
        super().__init__(params, defaults)
        for p in self.get_params():
            p.hess = 0.0
            self.state[p]["hessian step"] = 0

    def get_params(self):
        return (p for group in self.param_groups for p in group['params'] if p.requires_grad)

    def zero_hessian(self):
        for p in self.get_params():
            if not isinstance(p.hess, float) and self.state[p]["hessian step"] % self.lazy_hessian == 0:
                p.hess.zero_()

    @torch.no_grad()
    def set_hessian(self):
        params = []
        for p in filter(lambda p: p.grad is not None, self.get_params()):
            if self.state[p]["hessian step"] % self.lazy_hessian == 0:
                params.append(p)
            self.state[p]["hessian step"] += 1
        if len(params) == 0: return
        if self.generator.device != params[0].device:
            self.generator = torch.Generator(params[0].device).manual_seed(self.seed)
        grads = [p.grad for p in params]
        for i in range(self.n_samples):
            zs = [torch.randint(0, 2, p.size(), generator=self.generator, device=p.device) * 2.0 - 1.0 for p in params]
            h_zs = torch.autograd.grad(grads, params, grad_outputs=zs, only_inputs=True,
                                       retain_graph=i < self.n_samples - 1)
            for h_z, z, p in zip(h_zs, zs, params):
                p.hess += h_z * z / self.n_samples

    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None: loss = closure()
        self.zero_hessian()
        self.set_hessian()
        for group in self.param_groups:
            for p in group['params']:
                if p.grad is None or p.hess is None: continue
                p.mul_(1 - group['lr'] * group['weight_decay'])
                state = self.state[p]
                if len(state) == 1:
                    state['step'] = 0
                    state['exp_avg'] = torch.zeros_like(p.data)
                    state['exp_hessian_diag'] = torch.zeros_like(p.data)
                exp_avg, exp_hessian_diag = state['exp_avg'], state['exp_hessian_diag']
                beta1, beta2 = group['betas']
                state['step'] += 1
                exp_avg.mul_(beta1).add_(p.grad, alpha=1 - beta1)
                if (state['hessian step'] - 1) % self.lazy_hessian == 0:
                    exp_hessian_diag.mul_(beta2).add_(p.hess, alpha=1 - beta2)
                step_size = group['lr']
                denom = group['clip_threshold'] * exp_hessian_diag.clamp(0, None) + group['eps']
                ratio = (exp_avg.abs() / denom).clamp(None, 1)
                p.addcmul_(exp_avg.sign(), ratio, value=-step_size)
        return loss

print("✓ SophiaH defined")

✓ SophiaH defined


### 3c. Shampoo

In [ ]:
MAX_PRECOND_DIM = 512
MAX_PRECOND_SCALE = 50

def _matrix_power(matrix: torch.Tensor, power: float) -> torch.Tensor:
    device = matrix.device
    dim = matrix.size(0)
    if dim > MAX_PRECOND_DIM:
        diag = matrix.diagonal().clamp(min=1e-4)
        scaled = diag.pow(power).clamp(max=MAX_PRECOND_SCALE)
        return scaled.diag().to(device)
    matrix = matrix.cpu().double()
    matrix = 0.5 * (matrix + matrix.t())
    trace_val = matrix.trace().item()
    reg = max(1e-4, 0.01 * abs(trace_val) / dim)
    matrix.diagonal().add_(reg)
    try:
        eigvals, eigvecs = torch.linalg.eigh(matrix)
        eigvals = eigvals.clamp(min=reg)
        result = eigvecs @ eigvals.pow(power).diag() @ eigvecs.t()
        return result.float().to(device)
    except torch._C._LinAlgError:
        return torch.eye(dim, device=device)

class Shampoo(Optimizer):
    """Shampoo - Preconditioned Stochastic Tensor Optimization."""
    def __init__(self, params, lr=1e-1, momentum=0.0, weight_decay=0.0,
                 epsilon=1e-4, update_freq=1, precond_decay=0.99):
        defaults = dict(lr=lr, momentum=momentum, weight_decay=weight_decay,
                        epsilon=epsilon, update_freq=update_freq,
                        precond_decay=precond_decay)
        super().__init__(params, defaults)

    def step(self, closure=None):
        loss = None
        if closure is not None: loss = closure()
        for group in self.param_groups:
            decay = group["precond_decay"]
            momentum = group["momentum"]
            wd = group["weight_decay"]
            for p in group["params"]:
                if p.grad is None: continue
                grad = p.grad.data.clone()
                order = grad.ndimension()
                original_size = grad.size()
                state = self.state[p]
                if len(state) == 0:
                    state["step"] = 0
                    if momentum > 0:
                        state["momentum_buffer"] = torch.zeros_like(grad)
                    for dim_id, dim in enumerate(grad.size()):
                        state[f"precond_{dim_id}"] = group["epsilon"] * torch.eye(
                            dim, dtype=grad.dtype, device=grad.device)
                        state[f"inv_precond_{dim_id}"] = torch.eye(
                            dim, dtype=grad.dtype, device=grad.device)
                if wd > 0:
                    grad.add_(p.data, alpha=wd)
                if momentum > 0:
                    buf = state["momentum_buffer"]
                    buf.mul_(momentum).add_(grad, alpha=1.0 - momentum)
                    grad = buf.clone()
                update = grad
                for dim_id, dim in enumerate(grad.size()):
                    precond = state[f"precond_{dim_id}"]
                    inv_precond = state[f"inv_precond_{dim_id}"]
                    update = update.transpose(0, dim_id).contiguous()
                    transposed_size = update.size()
                    update = update.view(dim, -1)
                    update_t = update.t()
                    precond.mul_(decay).add_(update @ update_t, alpha=1.0 - decay)
                    if state["step"] % group["update_freq"] == 0:
                        inv_precond.copy_(_matrix_power(precond, -1.0 / order))
                    if dim_id == order - 1:
                        update = update_t @ inv_precond
                        update = update.view(original_size)
                    else:
                        update = inv_precond @ update
                        update = update.view(transposed_size)
                state["step"] += 1
                p.data.add_(update, alpha=-group["lr"])
        return loss

print("✓ Shampoo defined")

✓ Shampoo defined


### 3d. ASAM (Adaptive SAM)

In [ ]:
class ASAM(Optimizer):
    """Adaptive SAM - wraps a base optimizer with adaptive perturbation."""
    def __init__(self, params, base_optimizer_cls, rho=0.05, adaptive=True, **kwargs):
        defaults = dict(rho=rho, adaptive=adaptive, **kwargs)
        super().__init__(params, defaults)
        self.base_optimizer = base_optimizer_cls(self.param_groups, **kwargs)
        self.param_groups = self.base_optimizer.param_groups
        self.defaults.update(self.base_optimizer.defaults)

    @torch.no_grad()
    def first_step(self, zero_grad=False):
        grad_norm = self._grad_norm()
        for group in self.param_groups:
            scale = group["rho"] / (grad_norm + 1e-12)
            for p in group["params"]:
                if p.grad is None: continue
                self.state[p]["old_p"] = p.data.clone()
                e_w = (torch.pow(p, 2) if group["adaptive"] else 1.0) * p.grad * scale.to(p)
                p.add_(e_w)
        if zero_grad: self.zero_grad()

    @torch.no_grad()
    def second_step(self, zero_grad=False):
        for group in self.param_groups:
            for p in group["params"]:
                if p.grad is None: continue
                p.data = self.state[p]["old_p"]
        self.base_optimizer.step()
        if zero_grad: self.zero_grad()

    @torch.no_grad()
    def step(self, closure=None):
        assert closure is not None, "ASAM requires closure"
        closure = torch.enable_grad()(closure)
        self.first_step(zero_grad=True)
        closure()
        self.second_step()

    def _grad_norm(self):
        shared_device = self.param_groups[0]["params"][0].device
        norm = torch.norm(
            torch.stack([
                ((torch.abs(p) if group["adaptive"] else 1.0) * p.grad).norm(p=2).to(shared_device)
                for group in self.param_groups for p in group["params"]
                if p.grad is not None
            ]), p=2)
        return norm

    def load_state_dict(self, state_dict):
        super().load_state_dict(state_dict)
        self.base_optimizer.param_groups = self.param_groups

print("✓ ASAM defined")

✓ ASAM defined


### 3e. SASSHA

In [ ]:
class SASSHA(Optimizer):
    """SASSHA - SAM + Hutchinson Hessian trace."""
    def __init__(self, params, lr=0.15, betas=(0.9, 0.999), weight_decay=0.0,
                 rho=0.0, lazy_hessian=10, n_samples=1, perturb_eps=1e-12,
                 eps=1e-4, adaptive=False, hessian_power=1.0, seed=0, **kwargs):
        self.lazy_hessian = lazy_hessian
        self.n_samples = n_samples
        self.adaptive = adaptive
        self.seed = seed
        self.hessian_power_t = hessian_power
        defaults = dict(lr=lr, betas=betas, weight_decay=weight_decay,
                        rho=rho, perturb_eps=perturb_eps, eps=eps)
        super().__init__(params, defaults)
        for p in self.get_params():
            p.hess = 0.0
            self.state[p]["hessian step"] = 0
        self.generator = torch.Generator().manual_seed(self.seed)

    def get_params(self):
        return (p for group in self.param_groups for p in group['params'] if p.requires_grad)

    def zero_hessian(self):
        for p in self.get_params():
            if not isinstance(p.hess, float) and self.state[p]["hessian step"] % self.lazy_hessian == 0:
                p.hess.zero_()

    @torch.no_grad()
    def set_hessian(self):
        params = []
        for p in filter(lambda p: p.grad is not None, self.get_params()):
            if self.state[p]["hessian step"] % self.lazy_hessian == 0:
                params.append(p)
            self.state[p]["hessian step"] += 1
        if len(params) == 0: return
        if self.generator.device != params[0].device:
            self.generator = torch.Generator(params[0].device).manual_seed(self.seed)
        grads = [p.grad for p in params]
        for i in range(self.n_samples):
            zs = [torch.randint(0, 2, p.size(), generator=self.generator, device=p.device) * 2.0 - 1.0 for p in params]
            h_zs = torch.autograd.grad(grads, params, grad_outputs=zs, only_inputs=True,
                                       retain_graph=i < self.n_samples - 1)
            for h_z, z, p in zip(h_zs, zs, params):
                p.hess += h_z * z / self.n_samples

    @torch.no_grad()
    def perturb_weights(self, zero_grad=True):
        grad_norm = self._grad_norm()
        for group in self.param_groups:
            scale = group["rho"] / (grad_norm + group["perturb_eps"])
            for p in group["params"]:
                if p.grad is None: continue
                e_w = p.grad * scale.to(p)
                if self.adaptive: e_w *= torch.pow(p, 2)
                p.add_(e_w)
                self.state[p]['e_w'] = e_w
        if zero_grad: self.zero_grad()

    @torch.no_grad()
    def unperturb(self):
        for group in self.param_groups:
            for p in group['params']:
                if 'e_w' in self.state[p].keys():
                    p.data.sub_(self.state[p]['e_w'])

    @torch.no_grad()
    def _grad_norm(self, by=None):
        if not by:
            norm = torch.norm(torch.stack([
                ((torch.abs(p.data) if self.adaptive else 1.0) * p.grad).norm(p=2)
                for group in self.param_groups for p in group["params"] if p.grad is not None
            ]), p=2)
        else:
            norm = torch.norm(torch.stack([
                ((torch.abs(p.data) if self.adaptive else 1.0) * self.state[p][by]).norm(p=2)
                for group in self.param_groups for p in group["params"] if p.grad is not None
            ]), p=2)
        return norm

    @torch.no_grad()
    def step(self, closure=None, compute_hessian=True):
        loss = None
        if closure is not None: loss = closure()
        if compute_hessian:
            self.zero_hessian()
            self.set_hessian()
        k = self.hessian_power_t
        for group in self.param_groups:
            for p in group['params']:
                if p.grad is None or p.hess is None: continue
                if isinstance(p.hess, (int, float)):
                    p.hess = torch.zeros_like(p.data)
                else:
                    p.hess = p.hess.abs().clone()
                p.mul_(1 - group['lr'] * group['weight_decay'])
                state = self.state[p]
                if len(state) == 2:
                    state['step'] = 0
                    state['exp_avg'] = torch.zeros_like(p.data)
                    state['exp_hessian_diag'] = torch.zeros_like(p.data)
                    state['bias_correction2'] = 0
                exp_avg = state['exp_avg']
                exp_hessian_diag = state['exp_hessian_diag']
                beta1, beta2 = group['betas']
                state['step'] += 1
                exp_avg.mul_(beta1).add_(p.grad, alpha=1 - beta1)
                bias_correction1 = 1 - beta1 ** state['step']
                if (state['hessian step'] - 1) % self.lazy_hessian == 0:
                    exp_hessian_diag.mul_(beta2).add_(p.hess, alpha=1 - beta2)
                    bias_correction2 = 1 - beta2 ** state['step']
                    state['bias_correction2'] = bias_correction2 ** k
                step_size = group['lr'] / bias_correction1
                denom = ((exp_hessian_diag ** k) / max(state['bias_correction2'], 1e-12)).add_(group['eps'])
                p.addcdiv_(exp_avg, denom, value=-step_size)
        return loss

print("✓ SASSHA defined")

✓ SASSHA defined


## 4. Build Optimizer for RL

In [9]:
SEED = 42

def build_optimizer_rl(config, pol, vf):
    """Build the appropriate optimizer from config for both policy + value networks."""
    opt_type = config['optimizer']
    all_params = list(pol.parameters()) + list(vf.parameters())

    if opt_type == 'adam':
        return torch.optim.Adam(
            all_params, lr=config['lr'],
            betas=config.get('betas', (0.9, 0.999)),
            weight_decay=config.get('weight_decay', 0.0),
            eps=config.get('eps', 1e-8))

    elif opt_type == 'adahessian':
        return Adahessian(
            all_params, lr=config['lr'],
            betas=config.get('betas', (0.9, 0.999)),
            weight_decay=config.get('weight_decay', 0.0),
            eps=config.get('eps', 1e-4),
            hessian_power=config.get('hessian_power', 1.0),
            lazy_hessian=config.get('lazy_hessian', 10),
            n_samples=config.get('n_samples', 1), seed=SEED)

    elif opt_type == 'sophia':
        return SophiaH(
            all_params, lr=config['lr'],
            betas=config.get('betas', (0.965, 0.99)),
            weight_decay=config.get('weight_decay', 1e-4),
            eps=config.get('eps', 1e-4),
            clip_threshold=config.get('clip_threshold', 1.0),
            lazy_hessian=config.get('lazy_hessian', 10),
            n_samples=config.get('n_samples', 1), seed=SEED)

    elif opt_type == 'shampoo':
        return Shampoo(
            all_params, lr=config['lr'],
            momentum=config.get('momentum', 0.9),
            weight_decay=config.get('weight_decay', 0.0),
            epsilon=config.get('epsilon', 0.1),
            update_freq=config.get('update_freq', 50))

    elif opt_type == 'asam':
        return ASAM(
            all_params, torch.optim.SGD,
            rho=config.get('rho', 0.05), adaptive=True,
            lr=config['lr'],
            momentum=config.get('momentum', 0.9),
            weight_decay=config.get('weight_decay', 0.0))

    elif opt_type == 'sassha':
        return SASSHA(
            all_params, lr=config['lr'],
            betas=config.get('betas', (0.9, 0.999)),
            weight_decay=config.get('weight_decay', 0.0),
            rho=config.get('rho', 0.05),
            lazy_hessian=config.get('lazy_hessian', 10),
            n_samples=config.get('n_samples', 1),
            eps=config.get('eps', 1e-4),
            hessian_power=config.get('hessian_power', 1.0),
            seed=SEED)
    else:
        raise ValueError(f"Unknown optimizer: {opt_type}")

print("✓ build_optimizer_rl defined")

✓ build_optimizer_rl defined


## 5. PPO Learning with Second-Order Optimizers

In [10]:
def _needs_create_graph(config):
    """Does this optimizer need create_graph=True for Hessian?"""
    return config['optimizer'] in ('adahessian', 'sophia', 'sassha')

def _is_two_pass(config):
    """Does this optimizer use SAM-style two-pass?"""
    return config['optimizer'] in ('sassha', 'asam')


def ppo_learn(pol, vf, opt, buf, config, grad_clip=1.0):
    """
    PPO learning step with second-order optimizer support.
    Follows CBP paper's PPO implementation (lop/algos/rl/ppo.py).
    """
    g = config.get('g', 0.99)
    lm = config.get('lm', 0.95)
    bs = config.get('bs', 2048)
    n_itrs = config.get('n_itrs', 10)
    n_slices = config.get('n_slices', 16)
    clip_eps = config.get('clip_eps', 0.2)
    opt_type = config['optimizer']
    create_graph = _needs_create_graph(config)
    two_pass = _is_two_pass(config)

    os_t, acts, rs, op, logpbs, _, dones = buf.get(pol.dist_stack)

    # Compute values for GAE
    with torch.no_grad():
        pre_vals = vf.value(torch.cat((os_t, op)))

    # GAE computation
    vals = pre_vals.squeeze()
    rs_sq = rs.squeeze()
    dones_sq = dones.squeeze()
    advs = torch.zeros(len(rs_sq) + 1, device=rs_sq.device)
    for t in reversed(range(len(rs_sq))):
        delta = rs_sq[t] + (1 - dones_sq[t]) * g * vals[t + 1] - vals[t]
        advs[t] = delta + (1 - dones_sq[t]) * g * lm * advs[t + 1]
    v_rets = advs[:-1] + vals[:-1]
    advs = advs[:-1].view(-1, 1)
    advs = advs - advs.mean()
    if advs.std() != 0 and not torch.isnan(advs.std()):
        advs = advs / advs.std()
    v_rets = v_rets.view(-1, 1).detach()
    advs = advs.detach()

    inds = np.arange(os_t.shape[0])
    mini_bs = bs // n_slices
    p_loss_val, v_loss_val = 0.0, 0.0

    for _ in range(n_itrs):
        np.random.shuffle(inds)
        for start in range(0, len(os_t), mini_bs):
            ind = inds[start:start + mini_bs]

            if opt_type == 'sassha':
                # SASSHA two-pass: forward → backward → perturb → forward → backward(create_graph) → step
                opt.zero_grad()
                logpts, _ = pol.logp_dist(os_t[ind], acts[ind], to_log_features=True)
                grad_sub = (logpts - logpbs[ind]).exp()
                p_loss0 = -(grad_sub * advs[ind])
                ext_loss = -(torch.clamp(grad_sub, 1 - clip_eps, 1 + clip_eps) * advs[ind])
                p_loss = torch.max(p_loss0, ext_loss).mean()
                v_vals = vf.value(os_t[ind], to_log_features=True)
                v_loss = (v_rets[ind] - v_vals).pow(2).mean()
                loss = p_loss + v_loss
                loss.backward()
                opt.perturb_weights(zero_grad=True)
                # Second forward pass (perturbed)
                logpts2, _ = pol.logp_dist(os_t[ind], acts[ind])
                grad_sub2 = (logpts2 - logpbs[ind]).exp()
                p_loss2 = torch.max(-(grad_sub2 * advs[ind]),
                                    -(torch.clamp(grad_sub2, 1-clip_eps, 1+clip_eps) * advs[ind])).mean()
                v_vals2 = vf.value(os_t[ind])
                v_loss2 = (v_rets[ind] - v_vals2).pow(2).mean()
                loss2 = p_loss2 + v_loss2
                loss2.backward(create_graph=True)
                opt.unperturb()
                opt.step()
                opt.zero_grad(set_to_none=True)
                p_loss_val = p_loss.item()
                v_loss_val = v_loss.item()

            elif opt_type == 'asam':
                # ASAM two-pass with closure
                opt.zero_grad()
                logpts, _ = pol.logp_dist(os_t[ind], acts[ind], to_log_features=True)
                grad_sub = (logpts - logpbs[ind]).exp()
                p_loss0 = -(grad_sub * advs[ind])
                ext_loss = -(torch.clamp(grad_sub, 1 - clip_eps, 1 + clip_eps) * advs[ind])
                p_loss = torch.max(p_loss0, ext_loss).mean()
                v_vals = vf.value(os_t[ind], to_log_features=True)
                v_loss = (v_rets[ind] - v_vals).pow(2).mean()
                loss = p_loss + v_loss
                loss.backward()
                def closure():
                    logpts_c, _ = pol.logp_dist(os_t[ind], acts[ind])
                    gs = (logpts_c - logpbs[ind]).exp()
                    pl = torch.max(-(gs * advs[ind]),
                                   -(torch.clamp(gs, 1-clip_eps, 1+clip_eps) * advs[ind])).mean()
                    vv = vf.value(os_t[ind])
                    vl = (v_rets[ind] - vv).pow(2).mean()
                    total = pl + vl
                    total.backward()
                    return total
                opt.step(closure=closure)
                p_loss_val = p_loss.item()
                v_loss_val = v_loss.item()

            elif opt_type in ('adahessian', 'sophia'):
                # Hessian-based: backward with create_graph
                opt.zero_grad()
                logpts, _ = pol.logp_dist(os_t[ind], acts[ind], to_log_features=True)
                grad_sub = (logpts - logpbs[ind]).exp()
                p_loss0 = -(grad_sub * advs[ind])
                ext_loss = -(torch.clamp(grad_sub, 1 - clip_eps, 1 + clip_eps) * advs[ind])
                p_loss = torch.max(p_loss0, ext_loss).mean()
                v_vals = vf.value(os_t[ind], to_log_features=True)
                v_loss = (v_rets[ind] - v_vals).pow(2).mean()
                loss = p_loss + v_loss
                loss.backward(create_graph=True)
                if grad_clip > 0:
                    nn.utils.clip_grad_norm_(list(pol.parameters()) + list(vf.parameters()), grad_clip)
                opt.step()
                opt.zero_grad(set_to_none=True)
                p_loss_val = p_loss.item()
                v_loss_val = v_loss.item()

            else:
                # Standard (adam, shampoo): single pass
                opt.zero_grad()
                logpts, _ = pol.logp_dist(os_t[ind], acts[ind], to_log_features=True)
                grad_sub = (logpts - logpbs[ind]).exp()
                p_loss0 = -(grad_sub * advs[ind])
                ext_loss = -(torch.clamp(grad_sub, 1 - clip_eps, 1 + clip_eps) * advs[ind])
                p_loss = torch.max(p_loss0, ext_loss).mean()
                v_vals = vf.value(os_t[ind], to_log_features=True)
                v_loss = (v_rets[ind] - v_vals).pow(2).mean()
                loss = p_loss + v_loss
                loss.backward()
                if grad_clip > 0:
                    nn.utils.clip_grad_norm_(list(pol.parameters()) + list(vf.parameters()), grad_clip)
                opt.step()
                p_loss_val = p_loss.item()
                v_loss_val = v_loss.item()

    return {'p_loss': p_loss_val, 'v_loss': v_loss_val}


print("✓ PPO learning with second-order support defined")

✓ PPO learning with second-order support defined


## 6. Configs

In [11]:
# ── RL environment settings (from CBP paper: lop/rl/cfg/ant/cbp.yml) ──
ENV_NAME    = 'Ant-v4'
N_STEPS     = int(50e6)    # 10M steps (reduce for quick tests)
H_DIM       = (256, 256)
ACT_TYPE    = 'ReLU'
BS          = 2048
N_ITRS      = 10
N_SLICES    = 16
CLIP_EPS    = 0.2
GAMMA       = 0.99
LAMBDA      = 0.95
INIT        = 'lecun'
GRAD_CLIP   = 1.0

# ── SDP settings ──
SDP_GAMMA       = 0.3
SDP_INTERVAL    = 100_000   # Apply SDP every 100K steps (no task boundary in RL)
LOG_INTERVAL    = 1000      # Log metrics every 1K steps
SAVE_INTERVAL   = 500_000   # Save checkpoint every 500K steps

TIME_LIMIT_SECONDS = 11.5 * 3600

# ── Per-optimizer configs (with and without SDP) ──
CONFIGS = {
    # 0. Adam baseline + SDP
    'adam_sdp': dict(
        optimizer='adam',
        lr=0.0001, betas=(0.99, 0.99), weight_decay=1e-4, eps=1e-8,
        sdp_gamma=SDP_GAMMA,
    ),
    # 1. Adam baseline (no SDP)
    'adam_nosdp': dict(
        optimizer='adam',
        lr=0.0001, betas=(0.99, 0.99), weight_decay=1e-4, eps=1e-8,
        sdp_gamma=0.0,
    ),
    # 2. AdaHessian + SDP
    'adahessian_sdp': dict(
        optimizer='adahessian',
        lr=0.001, betas=(0.9, 0.999), weight_decay=1e-4, eps=1e-2,
        hessian_power=0.5, lazy_hessian=1, n_samples=1,
        sdp_gamma=SDP_GAMMA,
    ),
    # 3. AdaHessian (no SDP)
    'adahessian_nosdp': dict(
        optimizer='adahessian',
        lr=0.001, betas=(0.9, 0.999), weight_decay=1e-4, eps=1e-2,
        hessian_power=0.5, lazy_hessian=1, n_samples=1,
        sdp_gamma=0.0,
    ),
    # 4. SophiaH + SDP
    'sophia_sdp': dict(
        optimizer='sophia',
        lr=0.001, betas=(0.965, 0.99), weight_decay=1e-4, eps=1e-2,
        clip_threshold=0.04, lazy_hessian=1, n_samples=1,
        sdp_gamma=SDP_GAMMA,
    ),
    # 5. SophiaH (no SDP)
    'sophia_nosdp': dict(
        optimizer='sophia',
        lr=0.001, betas=(0.965, 0.99), weight_decay=1e-4, eps=1e-2,
        clip_threshold=0.04, lazy_hessian=1, n_samples=1,
        sdp_gamma=0.0,
    ),
    # 6. Shampoo + SDP
    'shampoo_sdp': dict(
        optimizer='shampoo',
        lr=0.001, momentum=0.9, weight_decay=3e-5, epsilon=0.1,
        update_freq=50,
        sdp_gamma=SDP_GAMMA,
    ),
    # 7. Shampoo (no SDP)
    'shampoo_nosdp': dict(
        optimizer='shampoo',
        lr=0.001, momentum=0.9, weight_decay=3e-5, epsilon=0.1,
        update_freq=50,
        sdp_gamma=0.0,
    ),
    # 8. ASAM + SDP
    'asam_sdp': dict(
        optimizer='asam',
        lr=0.001, rho=0.1, momentum=0.9, weight_decay=1e-4,
        sdp_gamma=SDP_GAMMA,
    ),
    # 9. ASAM (no SDP)
    'asam_nosdp': dict(
        optimizer='asam',
        lr=0.001, rho=0.1, momentum=0.9, weight_decay=1e-4,
        sdp_gamma=0.0,
    ),
    # 10. SASSHA + SDP
    'sassha_sdp': dict(
        optimizer='sassha',
        lr=0.001, betas=(0.9, 0.999), weight_decay=1e-4, rho=0.2,
        lazy_hessian=1, n_samples=1, eps=1e-3, hessian_power=0.5,
        sdp_gamma=SDP_GAMMA,
    ),
    # 11. SASSHA (no SDP)
    'sassha_nosdp': dict(
        optimizer='sassha',
        lr=0.001, betas=(0.9, 0.999), weight_decay=1e-4, rho=0.2,
        lazy_hessian=1, n_samples=1, eps=1e-3, hessian_power=0.5,
        sdp_gamma=0.0,
    ),
}

# ── Select which methods to run ──
METHODS_TO_RUN = [
    #'adam_sdp', 'adam_nosdp',
    #'adahessian_sdp', 
    #'adahessian_nosdp',
    #'sophia_sdp', 
    #'sophia_nosdp',
    #'shampoo_sdp', 'shampoo_nosdp',
    #'asam_sdp', 
    #'asam_nosdp',
    'sassha_sdp', 
    #'sassha_nosdp',
]

RESULTS_DIR = 'rl_results/secondorder_sdp'
CKPT_DIR    = os.path.join(RESULTS_DIR, 'checkpoints')
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

print(f"✓ Config: {ENV_NAME}, {N_STEPS:,} steps, bs={BS}, h_dim={H_DIM}")
print(f"  SDP: γ={SDP_GAMMA}, interval={SDP_INTERVAL:,} steps")
print(f"  Methods: {METHODS_TO_RUN}")

✓ Config: Ant-v4, 50,000,000 steps, bs=2048, h_dim=(256, 256)
  SDP: γ=0.3, interval=100,000 steps
  Methods: ['sassha_sdp']


## 7. Training Loop

In [ ]:
def run_method_rl(method_name, config):
    """
    Run PPO with a second-order optimizer ± SDP on Ant-v4.
    Follows CBP paper RL setup + rl-continual.ipynb structure.
    """
    opt_type = config['optimizer']
    sdp_gamma = config.get('sdp_gamma', 0.0)
    # Inject shared RL hyperparams into config
    config.setdefault('g', GAMMA)
    config.setdefault('lm', LAMBDA)
    config.setdefault('bs', BS)
    config.setdefault('n_itrs', N_ITRS)
    config.setdefault('n_slices', N_SLICES)
    config.setdefault('clip_eps', CLIP_EPS)

    print(f"\n{'='*70}")
    print(f"  {method_name} (optimizer={opt_type}) - PPO on {ENV_NAME}")
    if sdp_gamma > 0:
        print(f"  SDP enabled: γ={sdp_gamma}, interval={SDP_INTERVAL:,}")
    else:
        print(f"  SDP disabled")
    print(f"{'='*70}")

    wall_clock_start = time.time()
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

    # ── Create environment (Gymnasium API) ──
    env = gym.make(ENV_NAME)
    o_dim = env.observation_space.shape[0]
    a_dim = env.action_space.shape[0]
    print(f"  Env: {ENV_NAME}, obs_dim={o_dim}, act_dim={a_dim}")

    # ── Create networks (same as CBP paper) ──
    pol = MLPPolicy(o_dim, a_dim, act_type=ACT_TYPE, h_dim=list(H_DIM),
                    device=device, init=INIT)
    vf = MLPVF(o_dim, act_type=ACT_TYPE, h_dim=list(H_DIM),
               device=device, init=INIT)

    # ── Create optimizer ──
    opt = build_optimizer_rl(config, pol, vf)

    # ── Create buffer ──
    buf = Buffer(o_dim, a_dim, BS, device=device)

    # ── Results tracking ──
    R = {
        'episodic_returns': [], 'termination_steps': [],
        'dormant_units': [], 'weight_magnitude': [],
        'stable_rank': [], 'p_loss': [], 'v_loss': [],
        'sdp_cond': [],
        'hyperparams': {
            'seed': SEED, 'env_name': ENV_NAME, 'optimizer': opt_type,
            'sdp_gamma': sdp_gamma, 'lr': config['lr'],
        },
    }

    # ── Feature tracking for stable rank ──
    num_layers = len(H_DIM)
    short_term_feature_activity = torch.zeros(1000, H_DIM[-1], device=device)
    feature_idx = 0

    # ── Checkpoint resume ──
    ckpt_file = os.path.join(CKPT_DIR, f"ckpt_{method_name}.pt")
    start_step = 0
    if os.path.isfile(ckpt_file):
        print(f"  Loading checkpoint: {ckpt_file}")
        ckpt = torch.load(ckpt_file, map_location=device, weights_only=False)
        pol.load_state_dict(ckpt['pol'])
        vf.load_state_dict(ckpt['vf'])
        try:
            opt.load_state_dict(ckpt['opt'])
        except Exception:
            print("  Warning: could not restore optimizer state, starting fresh")
        R = ckpt.get('results', R)
        start_step = ckpt['step'] + 1
        feature_idx = ckpt.get('feature_idx', 0)
        # Reset Hessian state
        if hasattr(opt, 'get_params'):
            for p in opt.get_params():
                p.hess = 0.0
                opt.state[p]["hessian step"] = 0
        print(f"  ✓ Resumed from step {start_step:,}")
        del ckpt; torch.cuda.empty_cache()
    else:
        print("  (no checkpoint - training from scratch)")

    # ── Training loop (Gymnasium API) ──
    o, info = env.reset(seed=SEED)
    episode_return = 0
    episode_count = len(R['episodic_returns'])

    pbar = tqdm(range(start_step, N_STEPS), desc=method_name, initial=start_step, total=N_STEPS)
    for step in pbar:
        elapsed = time.time() - wall_clock_start
        if elapsed > TIME_LIMIT_SECONDS:
            print(f"\n  Time limit ({elapsed/3600:.1f}h). Saving checkpoint.")
            break

        # ── Get action and log features ──
        with torch.no_grad():
            a, logp, dist = pol.action(
                torch.tensor(o, dtype=torch.float32, device=device).unsqueeze(0),
                to_log_features=True
            )
            # Store last hidden layer features for stable rank
            if hasattr(pol, 'activations') and pol.activations:
                features = list(pol.activations.values())
                if len(features) >= 1 and features[-1] is not None:
                    feat = features[-1][0]  # last hidden layer
                    if feat.shape[0] == H_DIM[-1]:
                        short_term_feature_activity[feature_idx % 1000] = feat
                        feature_idx += 1

        a_np = a[0].cpu().numpy()

        # ── Step environment (Gymnasium API) ──
        op, r, terminated, truncated, info = env.step(a_np)
        done = terminated or truncated
        episode_return += r

        # ── Store transition ──
        buf.store(o, a_np, r, op, logp.cpu().numpy(),
                  pol.dist_to(dist, to_device='cpu'), float(done))
        o = op

        # ── Episode ended ──
        if done:
            R['episodic_returns'].append(episode_return)
            R['termination_steps'].append(step)
            episode_count += 1
            episode_return = 0
            o, info = env.reset()

        # ── Learning step when buffer is full ──
        if len(buf.o_buf) >= BS:
            learn_logs = ppo_learn(pol, vf, opt, buf, config, grad_clip=GRAD_CLIP)
            buf.clear()
            R['p_loss'].append(learn_logs['p_loss'])
            R['v_loss'].append(learn_logs['v_loss'])

            # Compute metrics
            R['weight_magnitude'].append(compute_weight_magnitude(pol, vf))
            R['dormant_units'].append(compute_dormant_units_rl(pol, vf))

        # ── Periodic SDP ──
        if sdp_gamma > 0 and step > 0 and step % SDP_INTERVAL == 0:
            cond_nums = apply_sdp_rl(pol, vf, sdp_gamma)
            avg_cond = sum(cond_nums) / max(len(cond_nums), 1) if cond_nums else 0.0
            R['sdp_cond'].append(avg_cond)
            print(f"\n  [{method_name}] Step {step:,}: SDP applied, avg cond={avg_cond:.1f}")

        # ── Compute stable rank every 10K steps ──
        if step > 0 and step % 10000 == 0:
            valid_samples = min(feature_idx, 1000)
            if valid_samples > 10:
                sr = compute_stable_rank_from_features(
                    short_term_feature_activity[:valid_samples].cpu())
                R['stable_rank'].append(sr)

        # ── Progress bar update ──
        if step > 0 and step % LOG_INTERVAL == 0:
            recent_rets = R['episodic_returns'][-10:] if R['episodic_returns'] else [0]
            avg_ret = np.mean(recent_rets)
            dormant = R['dormant_units'][-1] * 100 if R['dormant_units'] else 0
            sr_val = R['stable_rank'][-1] if R['stable_rank'] else H_DIM[-1]
            pbar.set_postfix({
                'Eps': episode_count,
                'Ret': f'{avg_ret:.0f}',
                'Dorm': f'{dormant:.1f}%',
                'SR': f'{sr_val:.0f}',
            })

        # ── Periodic checkpoint save ──
        if step > 0 and step % SAVE_INTERVAL == 0:
            ckpt_data = {
                'step': step, 'pol': pol.state_dict(), 'vf': vf.state_dict(),
                'opt': opt.state_dict(), 'results': R, 'feature_idx': feature_idx,
            }
            torch.save(ckpt_data, ckpt_file)
            elapsed = time.time() - wall_clock_start
            print(f"\n  [{method_name}] Checkpoint at step {step:,} [{elapsed/3600:.1f}h]")

    # ── Final save ──
    env.close()
    result_file = os.path.join(RESULTS_DIR, f'{method_name}_results.pkl')
    with open(result_file, 'wb') as f:
        pickle.dump(R, f)
    print(f"  ✓ {method_name}: {len(R['episodic_returns'])} episodes, "
          f"final avg return = {np.mean(R['episodic_returns'][-100:]):.1f}")
    return R


print("✓ Training loop defined")

✓ Training loop defined


## 8. Run All Experiments

In [13]:
all_results = {}
for method in METHODS_TO_RUN:
    cfg = CONFIGS[method]
    result = run_method_rl(method, cfg)
    all_results[method] = result


  sassha_sdp (optimizer=sassha) — PPO on Ant-v4
  SDP enabled: γ=0.3, interval=100,000


/usr/local/lib/python3.12/dist-packages/gymnasium/envs/registration.py:512: DeprecationWarning: WARN: The environment Ant-v4 is out of date. You should consider upgrading to version `v5`.
  logger.deprecation(


  Env: Ant-v4, obs_dim=27, act_dim=8
  (no checkpoint — training from scratch)


sassha_sdp:   0%|          | 1973/50000000 [00:02<13:30:44, 1027.83it/s, Eps=16, Ret=-1146, Dorm=0.0%, SR=256]/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:841: UserWarning: Using backward() with create_graph=True will create a reference cycle between the parameter and its gradient which can cause a memory leak. We recommend using autograd.grad when creating the graph to avoid this. If you have to use this function, make sure to reset the .grad fields of your parameters to None after use to break the cycle and avoid the leak. (Triggered internally at /pytorch/torch/csrc/autograd/engine.cpp:1291.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
sassha_sdp:   0%|          | 100120/50000000 [02:56<20:53:24, 663.52it/s, Eps=1061, Ret=-786, Dorm=1.8%, SR=256]


  [sassha_sdp] Step 100,000: SDP applied, avg cond=1032.9


...


  [sassha_sdp] Step 3,500,000: SDP applied, avg cond=6.7

  [sassha_sdp] Checkpoint at step 3,500,000 [1.7h]


sassha_sdp:   7%|▋         | 3600206/50000000 [1:42:43<11:58:36, 1076.15it/s, Eps=19068, Ret=-2052, Dorm=30.2%, SR=256]


  [sassha_sdp] Step 3,600,000: SDP applied, avg cond=17.0


sassha_sdp:   7%|▋         | 3690495/50000000 [1:45:10<21:59:51, 584.78it/s, Eps=19158, Ret=-28526, Dorm=39.6%, SR=256] 


ValueError: Expected parameter loc (Tensor of shape (128, 8)) of distribution Normal(loc: torch.Size([128, 8]), scale: torch.Size([128, 8])) to satisfy the constraint Real(), but found invalid values:
tensor([[nan, nan, nan,  ..., nan, nan, nan],
        [nan, nan, nan,  ..., nan, nan, nan],
        [nan, nan, nan,  ..., nan, nan, nan],
        ...,
        [nan, nan, nan,  ..., nan, nan, nan],
        [nan, nan, nan,  ..., nan, nan, nan],
        [nan, nan, nan,  ..., nan, nan, nan]], device='cuda:0',
       grad_fn=<AddmmBackward0>)

## 9. Results Plots

In [ ]:
METHOD_STYLES = {
    'adam_sdp':          {'color': '#607D8B', 'ls': '-',  'label': 'Adam+SDP'},
    'adam_nosdp':        {'color': '#607D8B', 'ls': '--', 'label': 'Adam'},
    'adahessian_sdp':   {'color': '#E91E63', 'ls': '-',  'label': 'AdaHessian+SDP'},
    'adahessian_nosdp': {'color': '#E91E63', 'ls': '--', 'label': 'AdaHessian'},
    'sophia_sdp':       {'color': '#9C27B0', 'ls': '-',  'label': 'SophiaH+SDP'},
    'sophia_nosdp':     {'color': '#9C27B0', 'ls': '--', 'label': 'SophiaH'},
    'shampoo_sdp':      {'color': '#FF9800', 'ls': '-',  'label': 'Shampoo+SDP'},
    'shampoo_nosdp':    {'color': '#FF9800', 'ls': '--', 'label': 'Shampoo'},
    'asam_sdp':         {'color': '#4CAF50', 'ls': '-',  'label': 'ASAM+SDP'},
    'asam_nosdp':       {'color': '#4CAF50', 'ls': '--', 'label': 'ASAM'},
    'sassha_sdp':       {'color': '#2196F3', 'ls': '-',  'label': 'SASSHA+SDP'},
    'sassha_nosdp':     {'color': '#2196F3', 'ls': '--', 'label': 'SASSHA'},
}

def _style(method):
    return METHOD_STYLES.get(method, {'color': 'gray', 'ls': '-', 'label': method})

def _clean(ax):
    ax.grid(True, alpha=0.25)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

def smooth_returns(rets, window=100):
    """Smooth episodic returns with a rolling average."""
    if len(rets) < window:
        return np.array(rets) if rets else np.array([0])
    return np.convolve(rets, np.ones(window)/window, mode='valid')


# ── Main comparison: 3×2 grid ──
fig, axes = plt.subplots(3, 2, figsize=(18, 18))
fig.suptitle(f'Second-Order Optimizers ± SDP — PPO on {ENV_NAME}\n'
             f'Network: {H_DIM}, {ACT_TYPE}, {N_STEPS:,} steps, bs={BS}',
             fontsize=14, fontweight='bold')

# ── (0,0) Episodic Return ──
ax = axes[0, 0]
for m, d in all_results.items():
    s = _style(m)
    sm = smooth_returns(d['episodic_returns'])
    ax.plot(np.arange(len(sm)), sm, color=s['color'], ls=s['ls'], lw=1.5,
            label=s['label'], alpha=0.9)
ax.set_xlabel('Episode'); ax.set_ylabel('Return')
ax.set_title('Episodic Return (smoothed)')
ax.legend(fontsize=7, ncol=2); _clean(ax)

# ── (0,1) Dormant Neurons ──
ax = axes[0, 1]
for m, d in all_results.items():
    s = _style(m)
    dorm = np.array(d['dormant_units']) * 100
    if len(dorm) > 0:
        ax.plot(np.arange(len(dorm)), dorm, color=s['color'], ls=s['ls'],
                lw=1.5, label=s['label'], alpha=0.9)
ax.set_xlabel('PPO Update'); ax.set_ylabel('Dormant (%)')
ax.set_title('Dormant Neuron Fraction')
ax.legend(fontsize=7, ncol=2); _clean(ax)

# ── (1,0) Stable Rank ──
ax = axes[1, 0]
for m, d in all_results.items():
    s = _style(m)
    sr = np.array(d['stable_rank'])
    if len(sr) > 0:
        ax.plot(np.arange(len(sr)), sr, color=s['color'], ls=s['ls'],
                lw=1.5, label=s['label'], alpha=0.9)
ax.set_xlabel('× 10K steps'); ax.set_ylabel('Stable Rank')
ax.set_title('Stable Rank (last hidden)')
ax.legend(fontsize=7, ncol=2); _clean(ax)

# ── (1,1) Weight Magnitude ──
ax = axes[1, 1]
for m, d in all_results.items():
    s = _style(m)
    wm = np.array(d['weight_magnitude'])
    if len(wm) > 0:
        ax.plot(np.arange(len(wm)), wm, color=s['color'], ls=s['ls'],
                lw=1.5, label=s['label'], alpha=0.9)
ax.set_xlabel('PPO Update'); ax.set_ylabel('Avg |W|')
ax.set_title('Average Weight Magnitude')
ax.legend(fontsize=7, ncol=2); _clean(ax)

# ── (2,0) Policy Loss ──
ax = axes[2, 0]
for m, d in all_results.items():
    s = _style(m)
    pl = np.array(d['p_loss'])
    if len(pl) > 0:
        sm = np.convolve(pl, np.ones(50)/50, mode='valid') if len(pl) > 50 else pl
        ax.plot(np.arange(len(sm)), sm, color=s['color'], ls=s['ls'],
                lw=1.5, label=s['label'], alpha=0.9)
ax.set_xlabel('PPO Update'); ax.set_ylabel('Policy Loss')
ax.set_title('Policy Loss (smoothed)')
ax.legend(fontsize=7, ncol=2); _clean(ax)

# ── (2,1) Value Loss ──
ax = axes[2, 1]
for m, d in all_results.items():
    s = _style(m)
    vl = np.array(d['v_loss'])
    if len(vl) > 0:
        sm = np.convolve(vl, np.ones(50)/50, mode='valid') if len(vl) > 50 else vl
        ax.plot(np.arange(len(sm)), sm, color=s['color'], ls=s['ls'],
                lw=1.5, label=s['label'], alpha=0.9)
ax.set_xlabel('PPO Update'); ax.set_ylabel('Value Loss')
ax.set_title('Value Loss (smoothed)')
ax.legend(fontsize=7, ncol=2); _clean(ax)

plt.tight_layout()
plot_file = os.path.join(RESULTS_DIR, 'secondorder_sdp_rl_comparison.png')
plt.savefig(plot_file, dpi=200, bbox_inches='tight')
plt.show()
print(f"✓ Main comparison plot saved to {plot_file}")

## 10. SDP Ablation: Δ(metric) = SDP − noSDP

In [ ]:
OPT_PAIRS = [
    ('adam_sdp',       'adam_nosdp',       'Adam',       '#607D8B'),
    ('adahessian_sdp', 'adahessian_nosdp', 'AdaHessian', '#E91E63'),
    ('sophia_sdp',     'sophia_nosdp',     'SophiaH',    '#9C27B0'),
    ('shampoo_sdp',    'shampoo_nosdp',    'Shampoo',    '#FF9800'),
    ('asam_sdp',       'asam_nosdp',       'ASAM',       '#4CAF50'),
    ('sassha_sdp',     'sassha_nosdp',     'SASSHA',     '#2196F3'),
]

fig_ab, axes_ab = plt.subplots(1, 3, figsize=(21, 5))
fig_ab.suptitle('SDP Ablation: Δ = (with SDP) − (without SDP) — RL (PPO)',
                fontsize=13, fontweight='bold')

for sdp_key, nosdp_key, label, color in OPT_PAIRS:
    if sdp_key not in all_results or nosdp_key not in all_results:
        continue
    d_sdp = all_results[sdp_key]
    d_no  = all_results[nosdp_key]

    # Δ Episodic Return (last N episodes aligned)
    n = min(len(d_sdp['episodic_returns']), len(d_no['episodic_returns']))
    if n > 100:
        sm_sdp = smooth_returns(d_sdp['episodic_returns'][:n], 100)
        sm_no  = smooth_returns(d_no['episodic_returns'][:n], 100)
        n2 = min(len(sm_sdp), len(sm_no))
        delta_ret = sm_sdp[:n2] - sm_no[:n2]
        axes_ab[0].plot(np.arange(len(delta_ret)), delta_ret, color=color,
                        lw=2, label=label, alpha=0.8)

    # Δ Stable Rank
    n_sr = min(len(d_sdp['stable_rank']), len(d_no['stable_rank']))
    if n_sr > 0:
        delta_sr = np.array(d_sdp['stable_rank'][:n_sr]) - np.array(d_no['stable_rank'][:n_sr])
        axes_ab[1].plot(np.arange(len(delta_sr)), delta_sr, color=color,
                        lw=2, label=label, alpha=0.8)

    # Δ Dormant Fraction
    n_d = min(len(d_sdp['dormant_units']), len(d_no['dormant_units']))
    if n_d > 0:
        delta_dorm = (np.array(d_sdp['dormant_units'][:n_d])
                      - np.array(d_no['dormant_units'][:n_d])) * 100
        axes_ab[2].plot(np.arange(len(delta_dorm)), delta_dorm, color=color,
                        lw=2, label=label, alpha=0.8)

for i, (title, ylabel) in enumerate([
    ('Δ Episodic Return', 'Δ Return'),
    ('Δ Stable Rank', 'Δ SR'),
    ('Δ Dormant Fraction', 'Δ Dormant (%)'),
]):
    axes_ab[i].set_title(title); axes_ab[i].set_xlabel('Index')
    axes_ab[i].set_ylabel(ylabel)
    axes_ab[i].axhline(0, color='black', ls=':', lw=0.8, alpha=0.5)
    axes_ab[i].legend(fontsize=9); _clean(axes_ab[i])

plt.tight_layout()
plot_abl = os.path.join(RESULTS_DIR, 'sdp_ablation_rl.png')
plt.savefig(plot_abl, dpi=200, bbox_inches='tight')
plt.show()
print(f"✓ SDP ablation plot saved to {plot_abl}")

## 11. Summary Table

In [ ]:
n_final = 100
print(f"\n{'='*90}")
print(f"  Second-Order Optimizers ± SDP — PPO on {ENV_NAME} - Final {n_final}-episode average")
print(f"{'='*90}")
header = (f"{'Method':<22} {'AvgReturn':>10} {'Dormant%':>10} "
          f"{'StableRank':>11} {'AvgW':>9} {'Episodes':>9}")
print(header)
print(f"{'─'*90}")

for method in METHODS_TO_RUN:
    if method not in all_results:
        continue
    d = all_results[method]
    s = _style(method)
    rets = d['episodic_returns']
    avg_ret = np.mean(rets[-n_final:]) if len(rets) >= n_final else np.mean(rets) if rets else 0
    dormant = (d['dormant_units'][-1] * 100) if d['dormant_units'] else 0
    sr = d['stable_rank'][-1] if d['stable_rank'] else 0
    wm = d['weight_magnitude'][-1] if d['weight_magnitude'] else 0
    n_eps = len(rets)
    print(f"  {s['label']:<20} {avg_ret:>10.1f} {dormant:>10.2f} "
          f"{sr:>11.1f} {wm:>9.4f} {n_eps:>9d}")

print(f"{'='*90}")